In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if not filename.lower().endswith(('.jpg', '.jpeg')):
            print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/industrial-surface-defect-detection/public/sample_submission.csv
/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv


In [2]:
ss = pd.read_csv("/kaggle/input/competitions/industrial-surface-defect-detection/public/sample_submission.csv")
train = pd.read_csv("/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv")

In [3]:
ss.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID           12000 non-null  int64 
 1   image_id     12000 non-null  object
 2   defect_type  12000 non-null  object
 3   mask_rle     12000 non-null  object
dtypes: int64(1), object(3)
memory usage: 375.1+ KB


In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45572 entries, 0 to 45571
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   image_id     45572 non-null  object
 1   defect_type  45572 non-null  object
 2   mask_rle     7201 non-null   object
 3   width        45572 non-null  int64 
 4   height       45572 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 1.7+ MB


In [5]:
train.head(5)

,image_id,defect_type,mask_rle,width,height
0,4S22I7SA34,defect_a,7255 6 7377 12 7505 12 7633 12 7762 11 7890 11...,800,128
1,4S22I7SA34,defect_b,NaN,800,128
2,4S22I7SA34,defect_c,NaN,800,128
3,4S22I7SA34,defect_d,NaN,800,128
4,DZ899111OE,defect_a,NaN,800,128


In [6]:
ss.head(5)

,ID,image_id,defect_type,mask_rle
0,1,WAFNKCRR7Q,defect_a,
1,2,WAFNKCRR7Q,defect_b,
2,3,WAFNKCRR7Q,defect_c,
3,4,WAFNKCRR7Q,defect_d,
4,5,YZXKEWSUDD,defect_a,


In [7]:
# submission = ss.copy()
# submission.to_csv("/kaggle/working/submission.csv", index=False)

In [8]:
df=train.copy()

In [9]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    images = [
        f for f in filenames
        if f.lower().endswith(('.jpg'))
    ]
    
    if images:
        print(dirname, len(images))

/kaggle/input/competitions/industrial-surface-defect-detection/public/test 3000
/kaggle/input/competitions/industrial-surface-defect-detection/public/train 11393


In [10]:
print(df['defect_type'].value_counts())

defect_type
defect_a    11393
defect_b    11393
defect_c    11393
defect_d    11393
Name: count, dtype: int64


In [11]:
print(df['image_id'].nunique())

11393


In [12]:
# each image id is having 4 defect types and its mask rle is require to be annoted.

In [13]:
print(df.groupby('defect_type')['mask_rle'].apply(lambda x: x.notna().sum()))

defect_type
defect_a    1081
defect_b     313
defect_c    4100
defect_d    1707
Name: mask_rle, dtype: int64


In [15]:
print(df[['width', 'height']].drop_duplicates())

   width  height
0    800     128


In [16]:
# all images have the same dimensions